# 23 · Finite-shot evaluation of the circuit models (inference only, no training)
**Runtime: T4 GPU** (about 1 h; resumable). Read-only for all run folders and data. Writes only `results/p2_shots/`.

The trained circuit models (`g_q`, `p_q`; both schedules; 5 folds x 3 seeds = 60 runs) are evaluated with the exact
expectation values (as in training) and with each circuit output replaced by a finite-shot estimate:
for every qubit expectation e, k ~ Binomial(N, (1 + e) / 2) and the estimate is 2k / N - 1, with N = 100, 1,000 and 10,000
shots and three draws per N. Each qubit is sampled independently (correct marginal variance; correlations between the
qubits of one measurement are not modelled). Draws are reproducible: a stateless generator seeded by a fixed constant per
(shot level, draw) and by the batch content. Datasets: CBIS-DDSM test fold of each run and the corrected DMID cohort (301 crops).
Shot effects are measured against exact predictions recomputed in this notebook under the same settings (predict without XLA); the stored Dice of notebooks 14 and 20 are compared as a check.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, gc
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
V2_DIR   = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
EV_CBIS  = f'{PROJECT}/results/p2_jit_eval/eval_per_image_all.csv'      # notebook 14 (exact, full precision)
EV_DMID  = f'{PROJECT}/results/p2_dmid_v2/eval'                         # notebook 20 (exact, full precision)
OUT      = f'{PROJECT}/results/p2_shots'
SETS     = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val'}
for s in SETS: os.makedirs(f'{OUT}/eval/{s}', exist_ok=True)
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)
import tensorflow as tf
import qseg.models, qseg2
from qseg.config import Config
cfg = Config(work_dir=OUT)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 1 · Finite-shot readout: wraps the exact circuit layer (global and patch-wise use the same class)
SHOTS = tf.Variable(0, dtype=tf.int32, trainable=False)      # 0 = exact expectation values
DRAW  = tf.Variable(0, dtype=tf.int64, trainable=False)      # seed part per (shot level, draw)
_exact_call = qseg.models.TFCircuitLayer.call
def _shot_call(self, inputs):
    e = _exact_call(self, inputs)
    def sample():
        n = tf.cast(SHOTS, tf.float32)
        p = tf.clip_by_value((1.0 + e) / 2.0, 0.0, 1.0)
        s2 = tf.math.floormod(tf.cast(tf.reduce_sum(tf.abs(inputs)) * 1000.0, tf.int64), 1_000_003)
        k = tf.random.stateless_binomial(tf.shape(e), seed=tf.stack([DRAW, s2]), counts=tf.fill(tf.shape(e), n), probs=p, output_dtype=tf.float32)
        return 2.0 * k / n - 1.0
    return tf.cond(SHOTS > 0, sample, lambda: e)
qseg.models.TFCircuitLayer.call = _shot_call
assert qseg2.PatchCircuitLayer.__mro__[1] is qseg.models.TFCircuitLayer
SHOT_LEVELS, N_DRAWS = (100, 1000, 10000), 3
def set_shots(n, d=0):
    SHOTS.assign(int(n)); DRAW.assign(int(1000 * (n > 0) + 10 * np.log10(max(n, 1)) + d))

In [ ]:
# 2 · Data (same arrays and order as notebooks 14 and 20)
meta = pd.read_csv(f'{V2_DIR}/metadata.csv')
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XD = np.stack([_png(f'{V2_DIR}/images/{r}.png', False) for r in meta.roi_id]); YD = np.stack([_png(f'{V2_DIR}/masks/{r}.png', True) for r in meta.roi_id])
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
ref_c = pd.read_csv(EV_CBIS, usecols=['set', 'model', 'fold', 'seed', 'dataset', 'roi_id', 'dice'])
ref_c = ref_c[(ref_c.dataset == 'CBIS') & ref_c.model.isin(['g_q', 'p_q'])]
print('DMID', XD.shape, '| CBIS', XC.shape, '| stored CBIS rows (circuit models):', len(ref_c))

In [ ]:
# 3 · Self-test on one run: exact reproduces the stored Dice; finite-shot circuit outputs lie on the 2k/N - 1 grid
def runs_of(root):
    out = []
    for d in sorted(glob.glob(f'{root}/*_fold*_seed*')):
        if not (os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')): continue
        s = json.load(open(f'{d}/summary.json'))
        if s['model'] in ('g_q', 'p_q'): out.append((d, s))
    return out
def load(d, s):
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(s['model'], (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{d}/best.weights.h5'); return nojit(m)
def nojit(model):
    try: model.jit_compile = False      # the shot sampler has a data-dependent count, so predict runs without XLA (exact and shots alike)
    except Exception as e: print('jit setting not changed:', e)
    return model
d0, s0 = runs_of(SETS['train'])[0]; m = load(d0, s0); set_shots(0)
_, _, te = qseg2.fold_split(DFC, int(s0['fold']))
P = m.predict(XC[te], batch_size=cfg.batch_size, verbose=0); dc = qseg2.per_image_scores(YC[te], P, cfg.threshold).dice.values
r = ref_c[(ref_c.set == 'train') & (ref_c.model == s0['model']) & (ref_c.fold == int(s0['fold'])) & (ref_c.seed == int(s0['seed']))].set_index('roi_id').dice
print(os.path.basename(d0), '| exact vs stored, max |diff|:', float(np.max(np.abs(dc - r.loc[DFC.roi_id.values[te]].values))))
cm = nojit(tf.keras.Model(m.input, m.get_layer('quantum_circuit').output))
for n in SHOT_LEVELS:
    set_shots(n); c = cm.predict(XC[te][:16], batch_size=cfg.batch_size, verbose=0).ravel()
    set_shots(0); c0 = cm.predict(XC[te][:16], batch_size=cfg.batch_size, verbose=0).ravel()
    k = (c + 1) * n / 2
    print(f'N={n:5d}: on grid {np.allclose(k, np.round(k), atol=1e-2)} | mean |shot - exact| {np.abs(c - c0).mean():.4f} (expected about {np.sqrt((1 - c0 ** 2).clip(0)).mean() / np.sqrt(n) * 0.8:.4f})')
set_shots(0); del m, cm; gc.collect()

In [ ]:
# 4 · All 60 circuit runs: exact + 3 shot levels x 3 draws, on the CBIS-DDSM test fold and the corrected DMID cohort (resumable)
def eval_run(set_name, d, s):
    dst = f'{OUT}/eval/{set_name}/{os.path.basename(d)}.csv'
    if os.path.exists(dst): return None
    m = load(d, s); _, _, te = qseg2.fold_split(DFC, int(s['fold'])); parts = []
    for ds, X, Y, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, meta.roi_id.values)):
        set_shots(0); P0 = m.predict(X, batch_size=cfg.batch_size, verbose=0)
        t = qseg2.per_image_scores(Y, P0, cfg.threshold)[['dice']].rename(columns={'dice': 'dice_exact'})
        t.insert(0, 'roi_id', ids); t.insert(0, 'dataset', ds); b0 = P0[..., 0] >= cfg.threshold
        for n in SHOT_LEVELS:
            for r in range(N_DRAWS):
                set_shots(n, r); Pn = m.predict(X, batch_size=cfg.batch_size, verbose=0)
                t[f'dice_s{n}_d{r}'] = qseg2.per_image_scores(Y, Pn, cfg.threshold).dice.values
                t[f'flip_s{n}_d{r}'] = ((Pn[..., 0] >= cfg.threshold) != b0).mean((1, 2))
                t[f'absdp_s{n}_d{r}'] = np.abs(Pn - P0)[..., 0].mean((1, 2))
        set_shots(0); parts.append(t)
    t = pd.concat(parts, ignore_index=True)
    t['model'], t['fold'], t['seed'], t['set'] = s['model'], int(s['fold']), int(s['seed']), set_name
    t.to_csv(dst, index=False, float_format='%.17g'); del m; gc.collect(); return t
n_new = 0
for set_name, root in SETS.items():
    rs = runs_of(root); print(set_name, len(rs), 'circuit runs')
    for d, s in rs:
        t0 = time.time(); t = eval_run(set_name, d, s)
        if t is not None:
            n_new += 1; g = t.groupby('dataset')
            print(f"{set_name:5s} {os.path.basename(d):20s} " + ' | '.join(f"{k} exact {v.dice_exact.mean():.4f} s100 {v.dice_s100_d0.mean():.4f} s1000 {v.dice_s1000_d0.mean():.4f}" for k, v in g) + f" ({time.time() - t0:.0f} s)", flush=True)
print('evaluated', n_new, 'new runs')

In [ ]:
# 5 · Combine + check against stored exact Dice + summary
allv = pd.concat([pd.read_csv(f) for s in SETS for f in sorted(glob.glob(f'{OUT}/eval/{s}/*.csv'))], ignore_index=True)
allv.to_csv(f'{OUT}/shots_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
rd = pd.concat([pd.read_csv(f, usecols=['model', 'fold', 'seed', 'set', 'roi_id', 'dice']) for s in SETS for f in glob.glob(f'{EV_DMID}/{s}/*.csv')]).assign(dataset='DMID')
ref = pd.concat([ref_c, rd[rd.model.isin(['g_q', 'p_q'])]], ignore_index=True)
chk = allv.merge(ref, on=['set', 'model', 'fold', 'seed', 'dataset', 'roi_id'], how='left')
print('runs:', allv.groupby(['set', 'model', 'fold', 'seed']).ngroups, '| rows:', len(allv), '| unmatched:', int(chk.dice.isna().sum()),
      '| max |exact - stored|:', float(np.nanmax(np.abs(chk.dice_exact - chk.dice))), '| images differing > 1e-6:', int((np.abs(chk.dice_exact - chk.dice) > 1e-6).sum()))
for n in SHOT_LEVELS:
    allv[f'delta_s{n}'] = allv[[f'dice_s{n}_d{r}' for r in range(N_DRAWS)]].mean(1) - allv.dice_exact
print(allv.groupby(['set', 'dataset', 'model'])[['dice_exact'] + [f'delta_s{n}' for n in SHOT_LEVELS]].mean().round(4).to_string())
print('DONE')

In [ ]:
# 6 · Auto-shutdown: save Drive, then release the GPU
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()